# 01 - Exploratory Data Analysis: Simulated Network Traffic

Goal: understand what normal traffic looks like and what *fingerprint* each attack leaves behind, **before** training any model. This is what a SOC data analyst does first.

> Run `python main.py all` once before opening this notebook so the data and model exist.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config
from src.data_loader import load_datasets
from src.preprocessing import clean_data

pd.set_option('display.max_columns', 30)
train, live = load_datasets('synthetic')
train.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,wrong_fragment,urgent,num_failed_logins,logged_in,count,srv_count,serror_rate,rerror_rate,same_srv_rate,diff_srv_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_serror_rate,timestamp,src_ip,dst_ip,label
0,0.0,tcp,http,SF,167.0,2342.0,0.0,0.0,0.0,1.0,10.0,10.0,0.01,0.09,0.77,0.08,127.0,169.0,0.81,0.04,2026-01-15 00:00:03.217808941,10.0.1.32,10.0.0.53,normal
1,1.0,icmp,eco_i,SF,64.0,0.0,0.0,0.0,0.0,1.0,10.0,9.0,0.00,0.06,0.89,0.04,204.0,92.0,0.70,0.00,2026-01-15 00:00:33.125150837,10.0.1.43,10.0.0.11,normal
2,1.0,tcp,smtp,SF,134.0,23559.0,0.0,0.0,0.0,1.0,8.0,8.0,0.00,0.03,0.62,0.08,141.0,226.0,0.88,0.01,2026-01-15 00:00:37.511914534,10.0.1.18,10.0.0.11,normal
3,0.0,udp,domain_u,SF,505.0,2626.0,0.0,0.0,0.0,1.0,114.0,90.0,0.48,0.02,0.82,0.08,32.0,158.0,0.77,0.08,2026-01-15 00:01:00.402887150,10.0.1.29,10.0.0.53,normal
4,1.0,tcp,smtp,SF,888.0,821.0,0.0,0.0,0.0,1.0,7.0,7.0,0.01,0.00,0.92,0.00,226.0,43.0,0.47,0.00,2026-01-15 00:01:01.652524851,10.0.1.12,10.0.0.25,normal


## 1. Size, columns and data types

In [2]:
print(train.shape)
train.dtypes

(27100, 24)


duration                         float64
protocol_type                        str
service                              str
flag                                 str
src_bytes                        float64
dst_bytes                        float64
wrong_fragment                   float64
urgent                           float64
num_failed_logins                float64
logged_in                        float64
count                            float64
srv_count                        float64
serror_rate                      float64
rerror_rate                      float64
same_srv_rate                    float64
diff_srv_rate                    float64
dst_host_count                   float64
dst_host_srv_count               float64
dst_host_same_srv_rate           float64
dst_host_serror_rate             float64
timestamp                 datetime64[ns]
src_ip                               str
dst_ip                               str
label                                str
dtype: object

## 2. Class balance
Attacks are the minority - the same imbalance real SOCs face. That is why we use `class_weight='balanced'` and look at precision/recall, not just accuracy.

In [3]:
train['label'].value_counts(normalize=True).round(3)

label
normal          0.738
dos             0.148
probe           0.059
brute_force     0.044
exfiltration    0.011
Name: proportion, dtype: float64

## 3. Attack fingerprints
Average value of key features per traffic type. Each attack stands out on different features:

In [4]:
key = ['src_bytes', 'dst_bytes', 'count', 'serror_rate', 'rerror_rate',
       'diff_srv_rate', 'num_failed_logins', 'duration']
train.groupby('label')[key].mean().round(2)

,src_bytes,dst_bytes,count,serror_rate,rerror_rate,diff_srv_rate,num_failed_logins,duration
label,,,,,,,,
brute_force,234.43,277.05,37.62,0.02,0.09,0.03,2.57,1.43
dos,19.13,110.38,213.99,0.86,0.02,0.04,0.00,0.01
exfiltration,3609893.94,754.99,4.20,0.02,0.02,0.05,0.00,410.28
normal,7713.10,8581.47,13.12,0.02,0.02,0.06,0.03,1.75
probe,302.81,178.15,50.18,0.28,0.65,0.74,0.00,0.06


* **dos** - huge `count`, `serror_rate` near 1 (half-open SYN connections), ~0 bytes
* **probe** - high `diff_srv_rate` + `rerror_rate` (touching many closed ports)
* **brute_force** - `num_failed_logins` > 0 on ssh/ftp/telnet
* **exfiltration** - enormous `src_bytes` (upload) and long `duration`

## 4. When do attacks happen?

In [5]:
train.assign(hour=pd.to_datetime(train['timestamp']).dt.hour)\
     .pivot_table(index='hour', columns='label', values='duration', aggfunc='size', fill_value=0)

label,brute_force,dos,exfiltration,normal,probe
hour,,,,,
0,0,0,0,175,0
1,0,0,0,173,0
2,0,0,0,161,1600
3,0,0,0,123,0
4,0,0,0,154,0
5,0,0,0,333,0
6,0,0,0,511,0
7,0,0,0,757,0
8,0,0,0,1273,0


## 5. Run the trained detector on a slice of live traffic

In [6]:
from src.detector import load_bundle, detect, generate_alerts
bundle = load_bundle()
results = detect(clean_data(live, verbose=False), bundle)
alerts = generate_alerts(results)
alerts[['alert_id','timestamp','severity','threat_type','src_ip','dst_ip','reason']].head(10)

,alert_id,timestamp,severity,threat_type,src_ip,dst_ip,reason
0,ALERT-000001,2026-01-16 00:39:22.996603921,MEDIUM,anomaly,10.0.1.70,10.0.0.20,Behaviour deviates from normal baseline (anoma...
1,ALERT-000002,2026-01-16 01:26:06.690535201,MEDIUM,anomaly,10.0.1.50,10.0.0.25,Behaviour deviates from normal baseline (anoma...
2,ALERT-000003,2026-01-16 02:00:00.352952924,HIGH,dos,213.129.211.194,10.0.0.10,Classifier matched 'dos' pattern (100% confide...
3,ALERT-000004,2026-01-16 02:00:01.412194165,HIGH,dos,149.248.154.145,10.0.0.10,Classifier matched 'dos' pattern (100% confide...
4,ALERT-000005,2026-01-16 02:00:01.935382990,HIGH,dos,49.251.188.240,10.0.0.10,Classifier matched 'dos' pattern (100% confide...
5,ALERT-000006,2026-01-16 02:00:04.822984327,HIGH,dos,149.248.154.145,10.0.0.10,Classifier matched 'dos' pattern (100% confide...
6,ALERT-000007,2026-01-16 02:00:09.542667701,HIGH,dos,149.248.154.145,10.0.0.10,Classifier matched 'dos' pattern (100% confide...
7,ALERT-000008,2026-01-16 02:00:10.119747552,HIGH,dos,119.31.88.130,10.0.0.10,Classifier matched 'dos' pattern (100% confide...
8,ALERT-000009,2026-01-16 02:00:10.952699931,HIGH,dos,119.31.88.130,10.0.0.10,Classifier matched 'dos' pattern (100% confide...
9,ALERT-000010,2026-01-16 02:00:12.056854685,HIGH,dos,12.28.219.55,10.0.0.10,Classifier matched 'dos' pattern (100% confide...


In [7]:
pd.crosstab(results['label'], results['verdict'], margins=True)

verdict,ANOMALY,BENIGN,KNOWN_ATTACK,All
label,,,,
anomaly,95,5,0,100
brute_force,0,9,591,600
dos,2,32,1966,2000
exfiltration,0,1,149,150
normal,109,9888,3,10000
probe,0,14,786,800
All,206,9949,3495,13650
